In [ ]:
import matplotlib
# matplotlib.use('nbagg')
import matplotlib.pyplot as plt
plt.rcParams["font.size"] = 15
import pandas as pd
import numpy as np
from collections import defaultdict
# %matplotlib nbagg

In [ ]:
physbo_dir = "physbo_data/"
nts_dir = "NTS_data/"

In [ ]:
prop_cycle = plt.rcParams['axes.prop_cycle']
colors = prop_cycle.by_key()['color']
mathods2marker = {"BBO_TS": ".", "NTS": "x"}
methods = ["Random", "BBO_TS", "DPP_TS", "NTS"]
prop_cycle = plt.rcParams['axes.prop_cycle']
colors = prop_cycle.by_key()['color']
method2color = {}
for i in range(len(methods)):
    method2color[methods[i]] = colors[i]
method2style = {
    "BBO_TS": "--",
    "NTS": ":",
}
method2color = {}
for i in range(len(methods)):
    method2color[methods[i]] = colors[i]
method2color = {
    "aggressive": "red",
    "moderate": "blue",
    "NTS": "green",
    "BBO_TS": "orange",
}
methods = ["BBO_TS", "NTS"]

# Max value plot by iteration

In [ ]:
import pathlib

fig, ax = plt.subplots()
features_list = []
for i, dir_path in enumerate([physbo_dir, nts_dir]):
    # Get the list of files under dir_path using pathlib
    path = pathlib.Path(dir_path)
    file_list = list(path.glob("*"))
    # Sort files in date order and process in a for loop
    file_list.sort()
    target_list = [-np.inf]
    features = []
    all_target = []
    th_history = []
    for file in file_list:
        # Read each CSV file: 1st column as action_id, columns 2 to second-to-last as feature_1~feature_N, last column as target
        df = pd.read_csv(file, header=None)
        action_id = df[0]
        feature = df.iloc[:, 1:-1]
        features.append(feature)
        target = df.iloc[:, -1]
        all_target.extend([e for e in target if pd.notnull(e)])
        th = np.percentile(np.array(all_target), 90)
        # print(th, all_target)
        th_history.append(max(th, max(th_history, default=0)))
        target_list.append(max([np.max(target_list), max(target)]))
    target_list = np.array(target_list)
    ax.plot(list(range(len(target_list[1:]))), target_list[1:], label=dir_path.strip("/"), color=method2color[methods[i]], marker=mathods2marker[methods[i]], linestyle=method2style[methods[i]])
    # if i == 1:
    #     ax.plot(list(range(len(th_history))), th_history, label="Threshold (conservative mode)", color="black", linestyle="--")
    features_list.append(features)
ax.legend(frameon=False, fontsize=15)
ax.set_ylabel("target", fontsize=15)
ax.set_xlabel("iteration", fontsize=15)
ax.set_xlim(0, 10)
ax.set_ylim(0.13, 0.18)
plt.savefig("nts_physbo_comparison_max.pdf", bbox_inches='tight')


In [ ]:
import pathlib

features_list = []
fig, ax = plt.subplots()
for i, dir_path in enumerate([physbo_dir, nts_dir]):
    # Get the list of files under dir_path using pathlib
    path = pathlib.Path(dir_path)
    file_list = list(path.glob("*"))
    # Sort files in date order and process in a for loop
    file_list.sort()
    target_list = [-np.inf]
    features = []
    all_target = []
    th_history = []
    for j, file in enumerate(file_list):
        # Read each CSV file: 1st column as action_id, columns 2 to second-to-last as feature_1~feature_N, last column as target
        df = pd.read_csv(file, header=None)
        action_id = df[0]
        feature = df.iloc[:, 1:-1]
        features.append(feature)
        target = df.iloc[:, -1]
        all_target.extend([e for e in target if pd.notnull(e)])
        th = np.percentile(np.array(all_target), 90)
        # print(th, all_target)
        th_history.append(max(th, max(th_history, default=0)))
        target_list.append(max([np.max(target_list), max(target)]))
        ax.scatter([j+i*0.2 for _ in range(len(target))], target, color=method2color[methods[i]], marker=mathods2marker[methods[i]], alpha=0.3, s=15)
    target_list = np.array(target_list)
    ax.plot(list(range(len(target_list[1:]))), target_list[1:], label=dir_path.strip("/"), color=method2color[methods[i]], marker=mathods2marker[methods[i]], linestyle=method2style[methods[i]])
    # if i == 1:
    #     ax.plot(list(range(len(th_history))), th_history, label="Threshold (conservative mode)", color="black", linestyle="--")
    features_list.append(features)
    # ax.legend(frameon=False, fontsize=15)
ax.set_ylabel("target", fontsize=15)
ax.set_xlabel("iteration", fontsize=15)
ax.set_xlim(0, 10)
ax.set_ylim(0, 0.19)
plt.savefig("nts_physbo_comparison_max_with_scatter.pdf", bbox_inches='tight')


In [ ]:
trg = target.to_numpy()
# violinplot from `trg`
fig, ax = plt.subplots()
ax.violinplot(trg[~np.isnan(trg)], positions=[0], widths=0.5, showmeans=False, showmedians=True)
ax.set_ylabel("target", fontsize=15)
ax.set_xlabel("NTS data", fontsize=15)

In [ ]:
from sklearn.metrics.pairwise import pairwise_distances
import networkx as nx

def maximal_independent_set(D, t):
    adj_matrix = np.array(D <= t, dtype=int)
    np.fill_diagonal(adj_matrix, 0)
    G = nx.from_numpy_array(np.array(adj_matrix))
    return nx.maximal_independent_set(G)

In [ ]:
all_target, all_features = defaultdict(list), defaultdict(list)
for i, dir_path in enumerate([physbo_dir, nts_dir]):
    # Get the list of files under dir_path using pathlib
    path = pathlib.Path(dir_path)
    file_list = list(path.glob("*"))
    # Sort files in date order and process in a for loop
    file_list.sort()
    for file in file_list:
        # Read each CSV file: 1st column as action_id, columns 2 to second-to-last as feature_1~feature_N, last column as target
        df = pd.read_csv(file, header=None)
        action_id = df[0]
        feature = df.iloc[:, 1:-1].to_numpy()
        target = df.iloc[:, -1].to_numpy()
        if len(all_target[i]) == 0 and len(all_features[i]) == 0:
            all_target[i].append(target)
            all_features[i].append(feature)
        else:
            # print(all_target[i][-1])
            all_target[i].append(np.concatenate((all_target[i][-1], target)))
            all_features[i].append(np.concatenate((all_features[i][-1], feature)))


In [ ]:
th = 0.14
t = 10
circles_above_th_dict = defaultdict(list)
for i in range(2):
    for feature, target in zip(all_features[i], all_target[i]):
        feature_above_th = np.array(feature)[np.array(target) > th]
        print(i, len(feature_above_th))
        D = pairwise_distances(feature_above_th, metric="euclidean")
        for _ in range(200):
            v = np.max([len(set(maximal_independent_set(D, t))) for _ in range(20)])
            # circles_above_th_dict[i].append(len(set(s)))
            if len(circles_above_th_dict[i]) == 0:
                circles_above_th_dict[i].append(v)
            elif circles_above_th_dict[i][-1] <= v:
                circles_above_th_dict[i].append(v)
                break

circles_above_th_dict

In [ ]:
fig, ax = plt.subplots()
methods = ["BBO_TS", "NTS"]
for i, method in enumerate(methods):
    # plt.plot(circles, label=methods[i], color=method2color[methods[i]])
    ax.plot(list(range(12)), circles_above_th_dict[i], label=method, color=method2color[methods[i]], marker=mathods2marker[methods[i]], linestyle=method2style[methods[i]])
ax.legend(frameon=False, fontsize=15)
ax.set_ylabel("number of circles", fontsize=15)
ax.set_xlabel("iteration", fontsize=15)
ax.set_xlim(0, 11)
ax.set_ylim(0, 70)
plt.savefig("circles_by_iteration.pdf", bbox_inches='tight')

In [ ]:
all_target, all_features = defaultdict(list), defaultdict(list)
for i, dir_path in enumerate([physbo_dir, nts_dir]):
    # Get the list of files under dir_path using pathlib
    path = pathlib.Path(dir_path)
    file_list = list(path.glob("*"))
    # Sort files in date order and process in a for loop
    file_list.sort()
    target_list = []
    features = []
    for file in file_list:
        # Read each CSV file: 1st column as action_id, columns 2 to second-to-last as feature_1~feature_N, last column as target
        df = pd.read_csv(file, header=None)
        action_id = df[0]
        feature = df.iloc[:, 1:-1].to_numpy()
        target = df.iloc[:, -1].to_numpy()
        for j in range(len(feature)):
            all_target[i].append(target[j])
            all_features[i].append(feature[j])
len(all_target[0]), len(all_target[1]), len(all_features[0]), len(all_features[1])


In [ ]:
# Visualize the distribution of samples exceeding the threshold in 2D using tSNE
import mplcursors
from sklearn.manifold import TSNE
fig, ax = plt.subplots(figsize=(8, 8))
th = 0.14
tsne = TSNE(n_components=2, random_state=0)
feature_above_th = list(np.array(all_features[0])[np.array(all_target[0]) > th])
N = len(feature_above_th)
feature_above_th += list(np.array(all_features[1])[np.array(all_target[1]) > th])
feature_above_th = np.array(feature_above_th)
X_embedded = tsne.fit_transform(feature_above_th)
for i in range(2):
    if i == 0:
        ax.scatter(X_embedded[:N, 0], X_embedded[:N, 1], label="physbo", marker='.', color=method2color[methods[i]], s=100)
    else:
        sc = ax.scatter(X_embedded[N:, 0], X_embedded[N:, 1], label="NTS", marker='x', color=method2color[methods[i]], s=100)
    # Separate areas where tSNE1 >= 10 and tSNE2 >= 10 with black lines
    # For tSNE1, draw lines only above points where tSNE2 = 10
    # For tSNE2, draw lines only to the right of points where tSNE1 = 10

# cursor = mplcursors.cursor(sc, hover=True)
# @cursor.connect("add")
# def _(sel):
#     sel.annotation.set_text(str(sel.index))  # sel.index is the index of the data point
# plt.show()
# ax.axvline(x=10, ymin=10, color='red', linestyle='--')
# ax.hlines(y=8, xmin=10, xmax=23, color='black', linestyle='--')
# ax.vlines(x=10, ymin=8, ymax=23, color='black', linestyle='--')
# ax.hlines(y=-3, xmin=13, xmax=23, color='black', linestyle='--')
# ax.vlines(x=13, ymax=-3, ymin=-23, color='black', linestyle='--')
# ax.hlines(y=8, xmin=-15, xmax=-10, color='black', linestyle='--')
# ax.hlines(y=-1, xmin=-23, xmax=-15, color='black', linestyle='--')
# ax.vlines(x=-10, ymax=8, ymin=23, color='black', linestyle='--')
# ax.vlines(x=-15, ymin=-1, ymax=8, color='black', linestyle='--')
# Display legend outside the figure
# ax.legend(frameon=False, fontsize=15, bbox_to_anchor=(1., 1), loc='upper left')
# ax.set_xlabel("tSNE1", fontsize=15)
# ax.set_ylabel("tSNE2", fontsize=15)
# ax.set_xlim(-23, 23)
# ax.set_ylim(-23, 23)
# Adjust figure layout with tight_layout
# plt.tight_layout()
plt.savefig(f"tsne_plot_{th}.pdf")


In [ ]:

# Relationship between Circles and distance threshold when target threshold is fixed at 0.14
# fig, ax = plt.subplots()
dist_threshold_list = np.arange(0, 16, 1)
N = 48 * 10
circles_auc_dict = defaultdict(list)
for th in [0.14]:
    all_target, all_features = defaultdict(list), defaultdict(list)
    circles_above_th_dict = defaultdict(list)
    for i, dir_path in enumerate([physbo_dir, nts_dir]):
        # Get the list of files under dir_path using pathlib
        path = pathlib.Path(dir_path)
        file_list = list(path.glob("*"))
        # Sort files in date order and process in a for loop
        file_list.sort()
        target_list = []
        features = []
        for file in file_list:
            # Read each CSV file: 1st column as action_id, columns 2 to second-to-last as feature_1~feature_N, last column as target
            df = pd.read_csv(file, header=None)
            action_id = df[0]
            feature = df.iloc[:, 1:-1].to_numpy()
            # Convert all values greater than 0 to 1
            feature[feature > 0] = 1
            target = df.iloc[:, -1].to_numpy()
            for j in range(len(feature)):
                all_target[i].append(target[j])
                all_features[i].append((feature[j] >= 0.5).astype(int))
    for t in dist_threshold_list:
        for i in range(2):
            # print(i, t)
            for _ in range(100):
                feature_above_th = np.array(all_features[i])[np.array(all_target[i]) > th]
                N = len(feature_above_th)
                D = pairwise_distances(feature_above_th, metric="hamming") * 48
                s = maximal_independent_set(D, t)
                v = len(set(s))
                # print(v)
                if len(circles_above_th_dict[i]) > 0:
                    if v <= circles_above_th_dict[i][-1]:
                        circles_above_th_dict[i].append(v)
                        break
                    else:
                        print(f"{i}, {t}, {v}, {circles_above_th_dict[i][-1]}")
                else:
                    circles_above_th_dict[i].append(v)
                    break
    print(f"th={th}")
    bbo_ts_auc = np.trapezoid(np.array(circles_above_th_dict[0]), dist_threshold_list)
    nts_auc = np.trapezoid(np.array(circles_above_th_dict[1]), dist_threshold_list)
    circles_auc_dict[0].append(bbo_ts_auc)
    circles_auc_dict[1].append(nts_auc)
    print(f"BBO TS, {bbo_ts_auc}")
    print(f"NTS, {nts_auc}")
    fig, ax = plt.subplots()
    ax.plot(dist_threshold_list, circles_above_th_dict[0], label="BBO_TS", color=method2color["BBO_TS"], marker=mathods2marker["BBO_TS"], linestyle=method2style["BBO_TS"])
    ax.plot(dist_threshold_list, circles_above_th_dict[1], label="NTS", color=method2color["NTS"], marker=mathods2marker["NTS"], linestyle=method2style["NTS"])
    ax.set_title(rf"Number of circles above threshold {th}" + '\n' + rf"(AUC: BBO_TS={bbo_ts_auc:.3f}, NTS={nts_auc:.3f})", fontsize=15)
    ax.set_xlim(min(dist_threshold_list), max(dist_threshold_list))
    ax.set_ylim(0)
    plt.savefig(f"circles_{th}.pdf", bbox_inches='tight')



In [ ]:
fig, ax = plt.subplots()
# circles_auc_dict を棒グラフで可視化
# 0.13, 0.14, 0.15 の3つの閾値に対して、BBO_TS と NTS の2つの手法を比較
th_list = [0.14, 0.15]
x = np.arange(len(th_list))  # the label locations
width = 0.35  # the width of the bars
rects1 = ax.bar(x - width/2, circles_auc_dict[0], width, label='BBO_TS', color=method2color["BBO_TS"])
rects2 = ax.bar(x + width/2, circles_auc_dict[1], width, label='NTS', color=method2color["NTS"])
# Add some text for labels, title and custom x-axis tick labels, etc.
ax.set_ylabel('Circles-AUC', fontsize=15)
ax.set_xlabel('target value threshold', fontsize=15)
ax.set_xticks(x)
ax.set_xticklabels(th_list)
# ax.legend(frameon=False, fontsize=15)
plt.savefig("circles_auc.pdf", bbox_inches='tight')

In [ ]:
fig, ax = plt.subplots()
ax.plot(dist_threshold_list, circles_above_th_dict[0], label="BBO_TS", color=method2color["BBO_TS"], marker=mathods2marker["BBO_TS"], linestyle=method2style["BBO_TS"])
ax.plot(dist_threshold_list, circles_above_th_dict[1], label="NTS", color=method2color["NTS"], marker=mathods2marker["NTS"], linestyle=method2style["NTS"])
ax.set_xlim(min(dist_threshold_list), max(dist_threshold_list))
# ax.set_ylim(0, 225)
ax.set_yscale('log')
plt.savefig("circles_log.pdf", bbox_inches='tight')